# Jour 2 -- Data manipulation -- SOLUTIONS (pédagogique)

Chaque exercice présente : la solution ; **POURQUOI CETTE APPROCHE** (+
alternatives) ; **PIÈGES / ERREURS FRÉQUENTES** ; **SCHÉMA À RECONNAÎTRE**.


In [ ]:
# ---------------------------------------------------------------------------
# Run this cell first. Paths are relative to this notebook's folder.
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

DATA = "../data"
drivers = pd.read_csv(f"{DATA}/drivers.csv")
trips = pd.read_csv(f"{DATA}/trips.csv")
print(drivers.shape, trips.shape)


---
## Exercice 1 -- `pd.to_datetime`, `.dt`

### Solution


In [ ]:
trips["trip_date"] = pd.to_datetime(trips["trip_date"])
trips["trip_month"] = trips["trip_date"].dt.month
trips["trip_weekday"] = trips["trip_date"].dt.day_name()

trips_per_month = trips["trip_month"].value_counts().sort_index()
print(trips["trip_date"].dtype)
print(trips_per_month)


**POURQUOI CONVERTIR EXPLICITEMENT.** Un CSV ne connaît que du texte --
pandas charge toujours une colonne de dates en `object` tant qu'on ne lui
dit pas explicitement `pd.to_datetime`. Sans cette conversion, `.dt` n'existe
pas et un tri sur la colonne se ferait par ordre **alphabétique** de la
chaîne, pas par ordre chronologique (ça fonctionne par coïncidence au
format `YYYY-MM-DD`, mais pas dans un format `DD/MM/YYYY`).

**L'ACCESSEUR `.dt`.** Une fois la colonne en `datetime64`, tous les
attributs temporels passent par `.dt` : `.dt.month`, `.dt.year`,
`.dt.day`, `.dt.day_name()`, `.dt.weekday` (0=lundi en entier),
`.dt.is_month_end`, etc. C'est l'équivalent de `.str` pour le texte : un
accesseur dédié au type de la colonne.

**PIÈGE FRÉQUENT.** Oublier de réaffecter
(`trips["trip_date"] = pd.to_datetime(...)`) -- `pd.to_datetime` ne
modifie jamais en place, elle renvoie toujours une nouvelle Series.

**SCHÉMA À RECONNAÎTRE.** *"colonne de date venant d'un CSV"* -> toujours
`pd.to_datetime` en tout premier, avant le moindre calcul temporel ou
tri.


---
## Exercice 2 -- Nettoyage `.str`

### Solution


In [ ]:
trips["dropoff_city_clean"] = trips["dropoff_city"].str.strip().str.title()
trips["mentions_client"] = trips["comment"].str.contains("client", case=False, na=False)

n_same_city = (trips["pickup_city"] == trips["dropoff_city_clean"]).sum()
print(trips["dropoff_city_clean"].unique())
print(trips["mentions_client"].isna().sum(), trips["mentions_client"].dtype)
print(n_same_city)


**POURQUOI CHAÎNER `.str.strip().str.title()`.** Chaque méthode `.str`
renvoie une nouvelle Series de texte, donc les méthodes s'enchaînent
comme des méthodes de `str` Python -- mais **vectorisées** sur toute la
colonne, ligne par ligne. `strip()` retire les espaces qui empêcheraient
`" paris"` de matcher `"Paris"` ; `title()` uniformise la casse en
`"Paris"`, `"Lyon"`, etc.

**`na=False` DANS `.str.contains`.** Sans ce paramètre, `.str.contains`
renvoie `NaN` (pas `False`) pour chaque ligne où `comment` est manquant --
un masque booléen avec des `NaN` dedans plante ou se comporte de façon
imprévisible dès qu'on l'utilise pour filtrer (`df[mask]`). `na=False`
force ces cas à `False`, ce qui est presque toujours le comportement
voulu ("un commentaire absent ne mentionne pas le client").

**`case=False` PLUTÔT QUE `.str.lower()` PRÉALABLE.** Les deux
fonctionnent, mais `case=False` évite une colonne intermédiaire inutile
quand le seul besoin est une comparaison insensible à la casse.

**PIÈGE FRÉQUENT.** Comparer `pickup_city` (déjà propre) à
`dropoff_city` **brute** (pas nettoyée) -- le compte de trajets
"même ville" serait alors artificiellement bas à cause des espaces/casse
incohérents, sans qu'aucune erreur ne soit levée. Un piège silencieux
typique des données textuelles.

**SCHÉMA À RECONNAÎTRE.** *"texte à comparer/joindre"* -> toujours
`strip()` + normalisation de casse (`lower`/`title`/`upper`) avant toute
comparaison ou merge sur cette colonne. *"chercher une sous-chaîne avec
des valeurs manquantes"* -> `.str.contains(..., na=False)`.


---
## Exercice 3 -- `groupby` + `agg` (une métrique)

### Solution


In [ ]:
avg_fare_by_city = trips.groupby("pickup_city")["fare_eur"].mean().sort_values(ascending=False)
trip_count_by_city = trips.groupby("pickup_city").size()

city_summary = pd.DataFrame({
    "avg_fare": trips.groupby("pickup_city")["fare_eur"].mean(),
    "n_trips": trips.groupby("pickup_city").size(),
})

print(avg_fare_by_city)
print(trip_count_by_city)
print(city_summary)


**LE PATTERN `groupby(...)[...].fonction()`.** `groupby("pickup_city")`
seul ne calcule rien -- c'est un objet `GroupBy` "en attente". Il faut
préciser **quelle colonne** agréger (`["fare_eur"]`) et **avec quelle
fonction** (`.mean()`) pour obtenir un résultat. C'est le triplet
toujours présent : clé de groupe, colonne cible, fonction d'agrégation.

**`.size()` vs `.count()`.** `size()` compte **toutes** les lignes du
groupe, y compris celles avec des `NaN` sur la colonne d'intérêt ;
`count()` (appelé sur une colonne précise) ignore les `NaN` de cette
colonne. Pour "nombre de courses", `size()` est le bon réflexe -- une
course reste une course même si `fare_eur` est manquant.

**PIÈGE FRÉQUENT.** Construire `city_summary` en recalculant deux fois le
même `groupby` (comme ci-dessus, pour rester lisible) au lieu d'utiliser
`.agg()` avec plusieurs fonctions d'un coup -- fonctionnellement correct,
mais l'exercice suivant montre la version plus propre (une seule passe).

**SCHÉMA À RECONNAÎTRE.** *"moyenne/somme/compte par catégorie"* ->
`df.groupby("categorie")["valeur"].fonction()`. *"nombre de lignes par
catégorie"* -> `.size()`, jamais `.count()` sauf besoin explicite
d'ignorer les manquants.


---
## Exercice 4 -- `groupby` + `agg` (plusieurs métriques)

### Solution


In [ ]:
city_stats = trips.groupby("pickup_city").agg(
    total_revenue=("fare_eur", "sum"),
    avg_distance=("distance_km", "mean"),
    n_trips=("trip_id", "count"),
)

city_status_avg = trips.groupby(["pickup_city", "status"])["fare_eur"].mean()
city_status_wide = city_status_avg.unstack()

print(city_stats)
print(city_status_avg)
print(city_status_wide)


**POURQUOI LA *NAMED AGGREGATION*.** `agg(nom=("col", "fonction"))`
calcule plusieurs statistiques (potentiellement sur des colonnes
différentes) en une seule passe sur les données, et nomme directement
chaque colonne de sortie -- pas de `MultiIndex` de colonnes à aplatir
après coup comme avec l'ancienne syntaxe `agg({"col": ["sum", "mean"]})`.
C'est la forme à utiliser par défaut dès que plusieurs métriques sont
demandées.

**GROUPER PAR PLUSIEURS CLÉS -> INDEX MULTI-NIVEAUX.**
`groupby(["pickup_city", "status"])` produit un résultat indexé par le
**couple** `(ville, statut)`. C'est exploitable tel quel (`.loc[("Paris",
"completed")]`), mais souvent plus lisible reformaté en tableau large.

**`.unstack()` POUR PASSER DE LONG À LARGE.** Le dernier niveau de
l'index (ici `status`) devient des colonnes -- transforme un résultat
"long" (une ligne par combinaison) en résultat "large" (une colonne par
valeur de `status`), plus proche d'un tableau croisé lisible à l'œil.

**PIÈGE FRÉQUENT.** Utiliser `("trip_id", "count")` en pensant comptez les
lignes -- correct ici car `trip_id` n'a pas de `NaN`, mais si la colonne
choisie avait des manquants, `count` les ignorerait silencieusement.
Préférer `("pickup_city", "size")` ou une colonne garantie sans `NaN`
quand le besoin est vraiment "nombre de lignes".

**SCHÉMA À RECONNAÎTRE.** *"plusieurs statistiques par catégorie, noms de
colonnes propres"* -> named aggregation. *"croiser deux catégories"* ->
`groupby([clé1, clé2])` puis `.unstack()` pour la lecture en tableau.


---
## Exercice 5 -- `merge` (inner vs left)

### Solution


In [ ]:
trips_inner = trips.merge(drivers, on="driver_id", how="inner")
trips_left = trips.merge(drivers, on="driver_id", how="left", suffixes=("_trip", "_driver"))

orphan_trips = trips_left[trips_left["age"].isna()]

print(len(trips), len(trips_inner), len(trips_left))
print(len(orphan_trips))
print(trips_left.columns.tolist())


**`how="inner"` NE GARDE QUE LES CLÉS PRÉSENTES DES DEUX CÔTÉS.** Les
courses dont le `driver_id` n'existe pas dans `drivers` disparaissent
silencieusement -- `len(trips_inner) < len(trips)`. C'est correct
**si l'objectif est d'analyser uniquement des courses avec chauffeur
connu**, mais dangereux si on ne s'attend pas à perdre des lignes : un
piège classique en entretien est de ne pas vérifier `len()` avant/après
un merge.

**`how="left"` PRÉSERVE TOUTES LES LIGNES DE LA TABLE DE GAUCHE.** Ici
`trips` reste la référence : chaque course apparaît exactement une fois
(en supposant `driver_id` unique côté `drivers`, ce qui est le cas après
le nettoyage du Jour 1), avec des `NaN` sur les colonnes de `drivers`
quand aucune correspondance n'existe.

**`suffixes=` QUAND DES NOMS DE COLONNES SE CHEVAUCHENT.** Si les deux
tables ont une colonne `status` (ou toute autre), pandas les renomme
automatiquement `status_trip`/`status_driver` (ou `_x`/`_y` par défaut) --
autant choisir des suffixes explicites plutôt que subir `_x`/`_y`,
illisibles dans un vrai livrable.

**PIÈGE FRÉQUENT.** Confondre "aucune ligne perdue" (`how="left"`,
`len()` inchangé) avec "aucune donnée manquante" -- un `left join` garde
toutes les lignes mais peut introduire de nouveaux `NaN`. Toujours
vérifier les deux à la fois : nombre de lignes ET présence de `NaN` sur
les colonnes ajoutées.

**SCHÉMA À RECONNAÎTRE.** *"je veux garder toutes les lignes de ma table
principale"* -> `how="left"` avec la table principale à gauche.
*"je ne veux que les correspondances valides des deux côtés"* ->
`how="inner"`. Toujours comparer `len()` avant/après pour savoir ce qui
s'est réellement passé.


---
## Exercice 6 -- `join` (sur l'index)

### Solution


In [ ]:
drivers_by_id = drivers.set_index("driver_id")
trips_by_driver = trips.set_index("driver_id")

joined = trips_by_driver.join(drivers_by_id[["rating"]], how="left")

print(len(joined) == len(trips))
print(joined.columns.tolist())
print(joined[["fare_eur", "rating"]].head())


**`join` = `merge` SPÉCIALISÉ SUR L'INDEX.** Sous le capot, `.join()`
appelle `merge` avec `left_index=True` (et `right_index=True` par
défaut) -- c'est un raccourci pratique quand les deux tables sont déjà
indexées par la clé commune, ce qui arrive souvent après un
`set_index` ou un `groupby` (dont le résultat est indexé par la clé de
groupe).

**POURQUOI SÉLECTIONNER `[["rating"]]` AVANT DE JOINDRE.** Sans ce filtre,
`.join()` ajouterait **toutes** les colonnes de `drivers_by_id`, y
compris celles déjà présentes dans `trips_by_driver` sous un autre nom --
risque de collision de noms. Ne joindre que la ou les colonnes réellement
utiles est plus sûr et plus lisible.

**QUAND PRÉFÉRER `join` À `merge`.** Si la clé est déjà l'index des deux
côtés : `join` est plus court à écrire. Dans tous les autres cas
(clé en colonne, clés de noms différents, jointure sur plusieurs
colonnes) : `merge`, plus explicite et plus flexible (`left_on`/
`right_on`, `how`, `suffixes`).

**PIÈGE FRÉQUENT.** Appeler `.join()` sur deux DataFrames dont l'un n'est
pas indexé par la clé commune -- lève une erreur ou, pire, joint sur
l'index numérique par défaut (0, 1, 2...) sans rapport avec `driver_id`,
produisant un résultat silencieusement faux.

**SCHÉMA À RECONNAÎTRE.** *"les deux tables sont déjà indexées par la même
clé"* -> `.join()`. *"la clé est une colonne normale"* -> `.merge(...,
on=...)`.


---
## Exercice 7 -- `concat`

### Solution


In [ ]:
trips_q1a = trips[trips["trip_month"].isin([1, 2])]
trips_q1b = trips[trips["trip_month"] == 3]

trips_recombined = pd.concat([trips_q1a, trips_q1b], ignore_index=True)
assert trips_recombined.shape == trips.shape

trips_with_source = pd.concat([trips_q1a, trips_q1b], keys=["jan_fev", "mars"])
print(trips_recombined.shape)
print(trips_with_source.index)


**`concat` EMPILE, `merge`/`join` COMBINENT CÔTE À CÔTE.** `pd.concat`
avec `axis=0` (par défaut) ajoute des **lignes** -- c'est l'outil pour
rassembler des morceaux d'un même schéma de colonnes (fichiers mensuels,
exports par région...), pas pour enrichir des colonnes à partir d'une clé
commune.

**`ignore_index=True`.** Sans ce paramètre, `trips_recombined` garderait
les index d'origine de `trips_q1a` et `trips_q1b` -- des doublons
d'index apparaîtraient (les deux morceaux viennent du même `trips`, donc
leurs index se chevauchent). `ignore_index=True` regénère un
`RangeIndex` propre de 0 à n-1.

**`keys=[...]` POUR TRACER L'ORIGINE.** Ajoute un niveau d'index
supplémentaire identifiant de quel morceau vient chaque ligne -- utile
pour déboguer ou pour un `groupby(level=0)` ultérieur sans avoir besoin
d'ajouter une colonne "source" à la main.

**PIÈGE FRÉQUENT.** Oublier `ignore_index=True` (ou `keys=`) puis
s'étonner de doublons d'index en aval -- typiquement un `.loc[3]` qui
renvoie soudain **plusieurs** lignes au lieu d'une seule après un
`concat` mal géré.

**SCHÉMA À RECONNAÎTRE.** *"recoller des lots de données au même schéma"*
-> `pd.concat([...], ignore_index=True)`. *"recoller en gardant une trace
du lot d'origine"* -> `pd.concat([...], keys=[...])`.


---
## Exercice 8 -- `apply` sur une colonne

### Solution


In [ ]:
def distance_bucket(km):
    if pd.isna(km):
        return "unknown"
    if km < 5:
        return "short"
    if km < 15:
        return "medium"
    return "long"


trips["distance_bucket"] = trips["distance_km"].apply(distance_bucket)


def price_per_km(row):
    if pd.isna(row["distance_km"]) or row["distance_km"] == 0:
        return np.nan
    return row["fare_eur"] / row["distance_km"]


trips["price_per_km"] = trips.apply(price_per_km, axis=1)

print(trips["distance_bucket"].value_counts(dropna=False))
print(trips["price_per_km"].isna().sum())
print(np.isinf(trips["price_per_km"]).sum())


**`apply` SUR UNE SEULE COLONNE (`Series.apply`) POUR UNE LOGIQUE
ARBITRAIRE.** Dès qu'une règle a plusieurs branches (`if`/`elif`/`else`)
qu'on ne peut pas exprimer facilement avec un seul `np.where`, une
fonction + `.apply()` reste la solution la plus lisible, quitte à être
plus lente que du code vectorisé pur.

**`pd.isna(km)` PLUTÔT QUE `km != km` OU `km is None`.** `pd.isna`
fonctionne uniformément que la valeur manquante soit `np.nan`, `None` ou
`pd.NaT` (dates) -- le réflexe universel pour tester un manquant, y
compris à l'intérieur d'une fonction appliquée valeur par valeur.

**`df.apply(fonction, axis=1)` POUR UNE LOGIQUE MULTI-COLONNES.** Quand
le calcul a besoin de **plusieurs colonnes de la même ligne** (ici
`distance_km` ET `fare_eur`), `axis=1` passe chaque **ligne entière**
(une `Series`) à la fonction -- on accède aux colonnes par
`row["nom_colonne"]`. C'est plus lent que `Series.apply` (une fonction
Python appelée par ligne, pas de vectorisation), à réserver aux cas où
une formulation vectorisée serait franchement moins lisible.

**PIÈGE FRÉQUENT.** Diviser directement `fare_eur / distance_km` sans
gérer `distance_km == 0` -- produit `inf` (pas une erreur, pas un `NaN`),
qui passe inaperçu dans `.isna().sum()` mais fausse ensuite n'importe
quelle moyenne ou tri sur la colonne.

**SCHÉMA À RECONNAÎTRE.** *"règle avec plusieurs branches sur une seule
colonne"* -> fonction + `Series.apply`. *"calcul qui combine plusieurs
colonnes de la même ligne, logique non triviale"* -> fonction +
`df.apply(axis=1)`, en gérant explicitement les cas limites (manquant,
division par zéro) à l'intérieur de la fonction.


---
## Exercice 9 -- `map`

### Solution


In [ ]:
zone_map = {"Paris": "Zone A", "Lyon": "Zone B", "Marseille": "Zone B", "Nice": "Zone C"}
trips["pickup_zone"] = trips["pickup_city"].map(zone_map)

n_unmapped = trips["pickup_zone"].isna().sum()
print(trips["pickup_zone"].value_counts(dropna=False))
print(n_unmapped)


**`map` = CORRESPONDANCE VALEUR -> VALEUR, RIEN DE PLUS.** `Series.map`
prend un dict (ou une autre Series) et remplace chaque valeur par sa
correspondance -- pas de condition, pas de logique, juste une table de
correspondance. C'est la version pandas d'un `VLOOKUP`/`RECHERCHEV` sur
une seule colonne.

**LES CLÉS ABSENTES DEVIENNENT `NaN`, SILENCIEUSEMENT.** Si une valeur de
`pickup_city` n'a pas d'entrée dans `zone_map`, `map` ne lève pas
d'erreur -- elle produit un `NaN` à cet endroit. C'est pourquoi on
vérifie systématiquement `.isna().sum()` après un `map` en contexte
métier réel : une grille de correspondance incomplète est un bug
silencieux typique.

**`map` VS `apply` VS `replace`.** `map` : dict simple, le plus rapide et
le plus lisible pour ce cas. `apply` : dès qu'il faut une condition ou un
calcul. `replace` : proche de `map` mais pensé pour ne remplacer que
certaines valeurs en laissant les autres inchangées (alors que `map`
transforme *tout*, avec `NaN` pour les valeurs non trouvées).

**PIÈGE FRÉQUENT.** Utiliser `map` en pensant qu'elle laisse les valeurs
non trouvées **inchangées** comme le ferait `replace` -- non, `map`
les remplace toutes par `NaN`. Si le besoin est "remplacer certaines
valeurs, garder les autres telles quelles", c'est `replace` qu'il faut.

**SCHÉMA À RECONNAÎTRE.** *"grille de correspondance fixe fournie
(dict/table)"* -> `.map(dict)`, avec vérification systématique des `NaN`
introduits pour les clés manquantes.


---
## Exercice 10 -- `np.where`

### Solution


In [ ]:
trips["is_long_trip"] = np.where(trips["distance_km"] > 15, True, False)

trips["fare_flag"] = np.where(
    trips["fare_eur"] < 15, "low",
    np.where(trips["fare_eur"] > 40, "high", "normal"),
)

print(trips["is_long_trip"].value_counts(dropna=False))
print(trips["fare_flag"].value_counts(dropna=False))


**`np.where` EST VECTORISÉ -- PAS DE FONCTION, PAS DE BOUCLE.** Il évalue
la condition sur **toute la colonne d'un coup** et choisit, élément par
élément, la valeur du deuxième ou du troisième argument. C'est nettement
plus rapide qu'un `.apply()` avec une fonction `if/else`, pour un besoin
strictement équivalent dans le cas à 2 branches.

**IMBRIQUER `np.where` POUR PLUS DE 2 BRANCHES.** Le troisième argument
d'un `np.where` peut lui-même être un autre `np.where` -- chaque niveau
d'imbrication ajoute une branche supplémentaire. Au-delà de 2-3 niveaux,
la lisibilité se dégrade vite : `pd.cut` (tranches numériques) ou une
fonction + `apply` deviennent alors préférables.

**COMPORTEMENT SUR `NaN` DANS LA CONDITION.** `distance_km > 15` avec un
`distance_km` manquant renvoie `False` (une comparaison avec `NaN` est
toujours `False` en pandas/numpy) -- donc `is_long_trip` vaut `False` pour
ces lignes, jamais `NaN`. C'est exactement le comportement demandé ici,
mais à vérifier systématiquement : ce n'est pas toujours le résultat
souhaité selon le contexte métier.

**PIÈGE FRÉQUENT.** Écrire une chaîne de `np.where` sans vérifier l'ordre
des conditions -- ici `fare_eur < 15` est testé en premier, donc une
valeur `NaN` (qui échoue `< 15`) retombe dans le `np.where` imbriqué, puis
échoue aussi `> 40`, et finit en `"normal"`. Si le besoin réel était un
flag `"unknown"` pour les manquants, il faudrait un test explicite
`pd.isna()` en amont.

**SCHÉMA À RECONNAÎTRE.** *"if/else simple sur une colonne entière, sans
fonction"* -> `np.where(cond, vrai, faux)`. *"plus de 2 branches"* ->
`np.where` imbriqué (jusqu'à 2-3 niveaux) ou `pd.cut`/fonction+`apply`
au-delà.


---
# RÉCAPITULATIF -- LES 15 RÉFLEXES DE LA SESSION

| Besoin | Instruction |
|---|---|
| convertir en date | `pd.to_datetime(df["col"])` |
| extraire mois/jour/nom du jour | `df["col"].dt.month` / `.dt.day_name()` |
| nettoyer du texte avant comparaison | `df["col"].str.strip().str.title()` |
| chercher une sous-chaîne (avec NaN) | `df["col"].str.contains("x", case=False, na=False)` |
| stat par catégorie | `df.groupby("k")["v"].mean()` / `.sum()` |
| nombre de lignes par catégorie | `df.groupby("k").size()` |
| plusieurs stats nommées | `df.groupby("k").agg(nom=("col", "fn"))` |
| croiser deux catégories en tableau large | `groupby([k1, k2])[...].mean().unstack()` |
| enrichir une table à partir d'une clé colonne | `df_a.merge(df_b, on="k", how=...)` |
| enrichir via un index déjà commun | `df_a.join(df_b[["col"]])` |
| empiler des lots de même schéma | `pd.concat([...], ignore_index=True)` |
| empiler en traçant l'origine | `pd.concat([...], keys=[...])` |
| logique arbitraire sur une colonne | `df["col"].apply(fonction)` |
| logique multi-colonnes par ligne | `df.apply(fonction, axis=1)` |
| correspondance simple valeur->valeur | `df["col"].map(dict)` |
| if/else vectorisé sans fonction | `np.where(cond, vrai, faux)` |

**Règle d'or :** avant d'écrire du code, posez-vous la question "est-ce
que j'empile des lignes (`concat`), est-ce que j'enrichis des colonnes à
partir d'une clé (`merge`/`join`), ou est-ce que je résume par groupe
(`groupby`)?" -- ce sont trois familles d'opérations différentes, et les
confondre est la source n°1 d'erreurs sur ce type d'exercice.

Si vous avez mis plus de 15 minutes pour tout le notebook, refaites-le
demain matin à froid, sans relire les solutions, avant de passer au
Jour 3.
